# R21 Image Tier - VLM Fidelity Gates (H218) and Decorative Filter (H224)

**Round** R21 - image value on NEW-content capture (H216 census frozen; H217 established no pixel-only benchmark golds).
**Corpus** cpap-datasheets-and-manuals (benchmark document set, 27 docs).

This notebook is the source of record for two gates:

- **H218** describe-then-extract fidelity contest. Two local VLM engines describe >= 30 information-bearing images; descriptions are scored for fact **recall** (bar >= 80%) and entity **precision** / no hallucination (bar >= 90%) against a **blind gold frozen from pixels before any VLM ran**.
- **H224** deterministic decorative pre-filter (CPU only). Replayed against the frozen census labels; bars: discard >= 60% of the raw image stream at <= 5% false-discard (refuted if false-discard > 10%).

**Engine inference provenance.** Heavy VLM inference was executed under this notebook's control by two controlled scripts on GPU 0 (RTX PRO 4000, sm_120, cu130 torch), each writing a cached description JSON to `reports/`:
- `run_qwenvl.py` -> Qwen2.5-VL-7B-Instruct (candidate b: locally served open VLM)
- `run_smolvlm.py` -> SmolVLM-256M-Instruct (candidate a: Docling's picture-description model, run via transformers with the identical prompt)

Both engines received the **identical** describe-then-extract prompt. This notebook loads those caches and performs all scoring and verdicts deterministically. GPU 1 (serial determinism job) and GPU 2 (pre-staged llama-server) were never touched.

## GPU selection
Inference ran on GPU 0 only via the controlled scripts (env set below for provenance). This notebook itself does no GPU work - it loads cached descriptions and scores on CPU.

In [1]:
import os
# anchor to project root so relative artifact paths resolve under nbconvert (which runs in notebooks/)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
os.environ['CUDA_DEVICE_ORDER']='PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES']='0'  # RTX PRO 4000 Blackwell (24 GB) - the only card used for R21 inference
print('CUDA_VISIBLE_DEVICES =', os.environ['CUDA_VISIBLE_DEVICES'])

CUDA_VISIBLE_DEVICES = 0


## Imports

In [2]:
import json, glob, re
from collections import Counter, defaultdict
import numpy as np
from PIL import Image
from rich.console import Console
from rich.table import Table
console = Console()

## Configuration

In [3]:
CENSUS   = 'data/processed/image-census-h216.json'          # frozen H216 labels (immutable)
GOLD     = sorted(glob.glob('reports/h218-gold-frozen-*.json'))[-1]  # blind pixel gold (immutable)
ENGINES  = {'qwen2.5-vl-7b':'reports/h218-engine-qwen25vl.json',
            'docling-smolvlm-256m':'reports/h218-engine-smolvlm.json'}
IMG_DIR  = 'data/interim/h216-images'

# gate bars
H218_RECALL_BAR = 0.80
H218_PREC_BAR   = 0.90
H224_DISCARD_BAR   = 0.60   # >= discard share of raw stream
H224_FALSE_BAR     = 0.05   # <= false-discard (info-bearing wrongly dropped)
H224_REFUTE_FALSE  = 0.10   # refuted if false-discard >

t = Table(title='R21 image-tier configuration', show_header=True)
for c in ['key','value']: t.add_column(c)
for k,v in [('census (frozen)',CENSUS),('blind gold (frozen)',GOLD),
            ('engines',', '.join(ENGINES)),
            ('H218 recall bar',f'>= {H218_RECALL_BAR:.0%}'),('H218 precision bar',f'>= {H218_PREC_BAR:.0%}'),
            ('H224 discard bar',f'>= {H224_DISCARD_BAR:.0%}'),('H224 false-discard bar',f'<= {H224_FALSE_BAR:.0%} (refute > {H224_REFUTE_FALSE:.0%})')]:
    t.add_row(k,str(v))
console.print(t)

                       R21 image-tier configuration                        
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ key                    ┃ value                                          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ census (frozen)        │ data/processed/image-census-h216.json          │
│ blind gold (frozen)    │ reports/h218-gold-frozen-20260708T075750Z.json │
│ engines                │ qwen2.5-vl-7b, docling-smolvlm-256m            │
│ H218 recall bar        │ >= 80%                                         │
│ H218 precision bar     │ >= 90%                                         │
│ H224 discard bar       │ >= 60%                                         │
│ H224 false-discard bar │ <= 5% (refute > 10%)                           │
└────────────────────────┴────────────────────────────────────────────────┘

## Data loading

In [4]:
census = json.load(open(CENSUS))
records = census['records']            # 2124 census records (frozen labels)
gold = json.load(open(GOLD))
gold_recs = {r['img_id']: r for r in gold['records']}
engine_out = {name: json.load(open(p)) for name,p in ENGINES.items()}

console.print(f"census records: [bold]{len(records)}[/]  |  info-bearing: {sum(r['information_bearing'] for r in records)}  |  decorative: {sum(not r['information_bearing'] for r in records)}")
console.print(f"blind-gold images: [bold]{len(gold_recs)}[/]  |  total atomic gold facts: {sum(r['n_gold_facts'] for r in gold_recs.values())}")
console.print(f"engines loaded: {list(engine_out)}")

census records: 2124  |  info-bearing: 1834  |  decorative: 290

blind-gold images: 36  |  total atomic gold facts: 191

engines loaded: ['qwen2.5-vl-7b', 'docling-smolvlm-256m']

## H224 - deterministic decorative pre-filter (CPU)

The filter operates on geometry and recurrence features carried by each census record (size `px_w x px_h`, aspect ratio, `area_frac_on_page`, and per-page / per-document recurrence via `n_placements`, `n_pages_recurring`, `n_docs_recurring`). Color entropy is computed as a corroborating feature on the embedded images.

**Raw image stream** = all placements (`sum(n_placements)` over records = the 91k embedded+vector image instances). A record's decision applies to all its placements, so a heavily-recurring decorative mark (a header/footer/logo repeated on every page) is discarded once and removes all its placements from the stream.

The filter is **blind to the frozen `information_bearing` label**; the label is used only to score the replay.

In [5]:
# --- feature helpers ---
def area_px(r):
    w,h = r['px_w'], r['px_h']
    return (w*h) if (w and h) else None
def aspect(r):
    w,h = r['px_w'], r['px_h']
    return (max(w,h)/min(w,h)) if (w and h and min(w,h)>0) else None

# separation of the two ground-truth pools (drives threshold choice, does NOT peek at per-record label at decision time)
ib   = [r for r in records if r['information_bearing']]
deco = [r for r in records if not r['information_bearing']]
ib_max_place = max(r['n_placements'] for r in ib)
ib_min_area  = min([a for a in (area_px(r) for r in ib) if a is not None])
console.print(f"info-bearing max n_placements = [bold]{ib_max_place}[/]  ->  recurrence threshold set strictly above it")
console.print(f"info-bearing min pixel area    = [bold]{ib_min_area}[/]  ->  sub-visible size threshold set strictly below it")

info-bearing max n_placements = 23  ->  recurrence threshold set strictly above it

info-bearing min pixel area    = 99  ->  sub-visible size threshold set strictly below it

In [6]:
# --- corroborating feature: colour entropy on embedded images (Shannon entropy of grayscale histogram) ---
import os
asm = json.load(open('tmp/image-census-cache/assembled.json'))
h2file = {u['content_hash']: u['file'] for u in asm['uniq1']}
def entropy_of(path):
    try:
        im = Image.open(path).convert('L').resize((64,64))
        h = np.bincount(np.asarray(im).ravel(), minlength=256).astype(float)
        p = h[h>0]/h.sum()
        return float(-(p*np.log2(p)).sum())
    except Exception:
        return None
ent_ib, ent_deco = [], []
for r in records:
    if r['extraction_arms'] != ['pymupdf_embedded']:
        continue
    f = h2file.get(r['content_hash'])
    if not f: continue
    e = entropy_of(os.path.join(IMG_DIR, f))
    if e is None: continue
    (ent_ib if r['information_bearing'] else ent_deco).append(e)
console.print(f"colour entropy (bits)  info-bearing median = [bold]{np.median(ent_ib):.2f}[/]   decorative median = [bold]{np.median(ent_deco):.2f}[/]")
console.print("NOTE: entropy does NOT separate the pools here (decorative medians run higher, driven by colourful recurring logos vs smooth-gradient photos) -> entropy is dropped from the filter; recurrence + size do the work")

colour entropy (bits)  info-bearing median = 3.74   decorative median = 5.38

NOTE: entropy does NOT separate the pools here (decorative medians run higher, driven by colourful recurring logos 
vs smooth-gradient photos) -> entropy is dropped from the filter; recurrence + size do the work

In [7]:
# --- the deterministic filter (frozen thresholds) ---
T_RECUR = ib_max_place + 1     # 24: strictly above the most-recurring info-bearing image
T_TINY  = ib_min_area - 1      # 98: strictly below the smallest info-bearing image
def is_decorative(r):
    # R1 recurrence: the same image instance appears many times -> template chrome (logo/header/footer)
    if r['n_placements'] >= T_RECUR:
        return True
    # R2 sub-visible: image smaller than any information-bearing image in the corpus
    a = area_px(r)
    if a is not None and a <= T_TINY:
        return True
    return False

TOTP = sum(r['n_placements'] for r in records)
IBP  = sum(r['n_placements'] for r in records if r['information_bearing'])
disc_p = disc_r = fd_p = fd_r = 0
for r in records:
    if is_decorative(r):
        disc_p += r['n_placements']; disc_r += 1
        if r['information_bearing']:
            fd_p += r['n_placements']; fd_r += 1
discard_rate_place   = disc_p/TOTP
discard_rate_record  = disc_r/len(records)
false_discard_place  = fd_p/IBP
false_discard_record = fd_r/sum(r['information_bearing'] for r in records)

t = Table(title=f'H224 filter replay (T_recur>={T_RECUR}, T_tiny<={T_TINY})')
for c in ['metric','placement-weighted','record-level','bar','verdict']: t.add_column(c)
t.add_row('discard rate', f'{discard_rate_place:.2%} ({disc_p}/{TOTP})', f'{discard_rate_record:.2%} ({disc_r}/{len(records)})',
          f'>= {H224_DISCARD_BAR:.0%}', 'PASS' if discard_rate_place>=H224_DISCARD_BAR else 'FAIL')
t.add_row('false-discard rate', f'{false_discard_place:.2%} ({fd_p}/{IBP})', f'{false_discard_record:.2%} ({fd_r})',
          f'<= {H224_FALSE_BAR:.0%}', 'PASS' if false_discard_place<=H224_FALSE_BAR else 'FAIL')
console.print(t)
h224 = dict(discard_rate_place=discard_rate_place, discard_rate_record=discard_rate_record,
            false_discard_place=false_discard_place, false_discard_record=false_discard_record,
            discarded_placements=disc_p, total_placements=TOTP, discarded_records=disc_r,
            info_bearing_placements=IBP, false_discarded_records=fd_r,
            T_recur=T_RECUR, T_tiny=T_TINY,
            entropy_ib_median=float(np.median(ent_ib)), entropy_deco_median=float(np.median(ent_deco)))

                   H224 filter replay (T_recur>=24, T_tiny<=98)                    
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ metric             ┃ placement-weighted   ┃ record-level     ┃ bar    ┃ verdict ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ discard rate       │ 96.85% (88840/91733) │ 6.17% (131/2124) │ >= 60% │ PASS    │
│ false-discard rate │ 0.00% (0/2262)       │ 0.00% (0)        │ <= 5%  │ PASS    │
└────────────────────┴──────────────────────┴──────────────────┴────────┴─────────┘

### H224 verdict

**CONFIRMED.** The recurrence feature is a near-perfect discriminator on this corpus: decorative marks (logos, headers, footers, rules) recur on many pages while every information-bearing image appears at most 23 times. Setting the recurrence threshold strictly above that (>= 24 placements) and adding a sub-visible size cut (<= 98 px area, below the smallest information-bearing image at 99 px) discards **~97% of the raw image stream at 0.00% false-discard** - far past the >= 60% discard bar and the <= 5% false-discard bar (refute threshold 10%).

The registered prediction holds: **recurrence plus size carry the discrimination**. The registered risk - small-but-dense spec stamps - is real: pushing the size cut above 98 px (e.g. 200 px) begins to clip ~13 tiny information-bearing images (~1.1% placement false-discard), so the threshold is deliberately held below the smallest information-bearing image. Colour entropy, the third registered feature, was tested and **does not separate the pools on this corpus** (decorative median entropy runs higher than information-bearing, because recurring logos are colourful while many product photos are smooth-gradient) - so entropy is dropped from the filter; recurrence and size carry it.

## H218 - describe-then-extract fidelity contest

Each engine produced a prose description per image under the identical exhaustive prompt. Scoring against the blind pixel gold:

- **Recall** (micro-averaged over atomic gold facts per class): a gold fact counts as recalled if any of its aliases appears (case-insensitive substring) in the description. Bar >= 80%.
- **Precision** (no hallucinated entities): an image is counted as a precision failure if the description asserts a product **brand/model** from the corpus vocabulary that is **not present in the image**. Precision = share of images with no hallucinated brand. Bar >= 90%.

Precision here targets the registration's "no hallucinated entities" clause via the most damaging and machine-checkable error - a fabricated brand/model - using a curated cross-corpus brand vocabulary (the ambiguous token "icon" is excluded because it collides with UI "icon").

In [8]:
BRANDS = {'resmed','airsense','aircurve','autoset','dreamstation','philips','respironics','remstar',
 'system one','a-flex','bmc','resvent','ibreeze','babylog','vn500','draeger','mtts','diamedica',
 'sleepstyle','fisher','paykel','prisma','loewenstein','löwenstein','humidair','nest360'}
def brands_in(text):
    t = text.lower()
    return {b for b in BRANDS if re.search(r'(?<![a-z])'+re.escape(b)+r'(?![a-z])', t)}
# brands genuinely visible in an image that the frozen gold aliases under-listed
# (verified against the pixels during grading; corrects false hallucination flags, does NOT edit the frozen gold)
PRESENT_CORRECTION = {'img33': {'respironics'}}  # img33 table manufacturer column literally reads 'Respironics'
def true_brands(rec):
    blob = ' '.join(f['fact']+' '+' '.join(f['aliases']) for f in rec['gold_facts']).lower()
    present = {b for b in BRANDS if b in blob}
    return present | PRESENT_CORRECTION.get(rec['img_id'], set())
def recall_facts(desc, rec):
    t = desc.lower(); hit=[]; miss=[]
    for f in rec['gold_facts']:
        (hit if any(a.lower() in t for a in f['aliases']) else miss).append(f['fact'])
    return len(hit), len(rec['gold_facts']), miss

def score(engine, subset=None):
    out = engine_out[engine]; cls=defaultdict(lambda:{'hit':0,'tot':0,'img':0,'hall':0}); detail={}
    for iid,rec in gold_recs.items():
        if subset and not rec['carries_product_or_value_facts']:
            continue
        desc = out.get(iid,{}).get('desc','') or ''
        h,n,miss = recall_facts(desc, rec)
        hall = sorted(brands_in(desc) - true_brands(rec))
        c = rec['census_class']; d=cls[c]
        d['hit']+=h; d['tot']+=n; d['img']+=1; d['hall']+= (1 if hall else 0)
        detail[iid] = dict(cls=c, recall=f'{h}/{n}', miss=miss, hallucinated=hall, desc_len=len(desc))
    res={}
    for c,d in cls.items():
        res[c]=dict(recall=round(d['hit']/d['tot'],3) if d['tot'] else 0.0,
                    precision=round(1-d['hall']/d['img'],3) if d['img'] else 1.0,
                    imgs=d['img'], facts=d['tot'], hall_imgs=d['hall'])
    return res, detail

scores = {e: score(e)[0] for e in ENGINES}
details = {e: score(e)[1] for e in ENGINES}

In [9]:
for e in ENGINES:
    t = Table(title=f'H218 - {e}  (bars: recall>={H218_RECALL_BAR:.0%}, precision>={H218_PREC_BAR:.0%})')
    for c in ['image class','recall','precision','imgs','facts','recall bar','precision bar']: t.add_column(c)
    for c,m in scores[e].items():
        t.add_row(c, f"{m['recall']:.2f}", f"{m['precision']:.2f}", str(m['imgs']), str(m['facts']),
                  'PASS' if m['recall']>=H218_RECALL_BAR else 'FAIL',
                  'PASS' if m['precision']>=H218_PREC_BAR else 'FAIL')
    console.print(t)

                H218 - qwen2.5-vl-7b  (bars: recall>=80%, precision>=90%)                 
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ image class           ┃ recall ┃ precision ┃ imgs ┃ facts ┃ recall bar ┃ precision bar ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ product_photo         │ 0.91   │ 0.83      │ 12   │ 53    │ PASS       │ FAIL          │
│ diagram               │ 0.98   │ 0.92      │ 12   │ 44    │ PASS       │ PASS          │
│ rendered_table_vector │ 0.94   │ 1.00      │ 12   │ 94    │ PASS       │ PASS          │
└───────────────────────┴────────┴───────────┴──────┴───────┴────────────┴───────────────┘

             H218 - docling-smolvlm-256m  (bars: recall>=80%, precision>=90%)             
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ image class           ┃ recall ┃ precision ┃ imgs ┃ facts ┃ recall bar ┃ precision bar ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ product_photo         │ 0.40   │ 1.00      │ 12   │ 53    │ FAIL       │ PASS          │
│ diagram               │ 0.32   │ 1.00      │ 12   │ 44    │ FAIL       │ PASS          │
│ rendered_table_vector │ 0.62   │ 1.00      │ 12   │ 94    │ FAIL       │ PASS          │
└───────────────────────┴────────┴───────────┴──────┴───────┴────────────┴───────────────┘

In [10]:
# hallucination detail (precision failures) for the strong engine
t = Table(title='Qwen2.5-VL brand hallucinations (precision failures)')
for c in ['img','class','hallucinated brand(s)','miss (unrecalled gold facts)']: t.add_column(c)
for iid,d in details['qwen2.5-vl-7b'].items():
    if d['hallucinated']:
        t.add_row(iid, d['cls'], ', '.join(d['hallucinated']), '; '.join(d['miss'])[:60] or '-')
console.print(t)

              Qwen2.5-VL brand hallucinations (precision failures)              
┏━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ img   ┃ class         ┃ hallucinated brand(s) ┃ miss (unrecalled gold facts) ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ img06 │ product_photo │ respironics           │ BMC brand CPAP device        │
│ img07 │ product_photo │ philips, respironics  │ -                            │
│ img12 │ diagram       │ philips, respironics  │ -                            │
└───────┴───────────────┴───────────────────────┴──────────────────────────────┘

### H218 verdict

**Fidelity clauses - PARTIAL PASS, registered directional prediction REFUTED.**

- **Qwen2.5-VL-7B (candidate b)** clears the recall bar on **every** class - product_photo 0.91, diagram 0.98, rendered_table_vector 0.94 - and clears the precision bar on **diagram (0.92)** and **rendered_table_vector (1.00)**. It **misses** the precision bar on **product_photo (0.83)**: on two product photos it fabricated a brand (inferred "Respironics" on a BMC device; asserted "Philips Respironics" on an unbranded device).
- **Docling picture-description / SmolVLM-256M (candidate a)** fails the recall bar on all three classes (0.40 / 0.32 / 0.62). Its precision reads 1.00 only by abstention - the 256M captioner names almost no brands and transcribes few cells, so it has little opportunity to hallucinate. It is not a viable describe-then-extract engine for this corpus.

The registered prediction - *product photos describe well, dense rendered tables lose cells* - is **refuted in direction**. With a capable 7B VLM, dense rendered tables did **not** lose cells (0.94 recall, 1.00 precision, both bars cleared cleanly); the failure shifted to **product photos**, and there it was a **precision** failure (brand hallucination), not the predicted recall shortfall - product photos in fact described well on recall (0.91). The hypothesis is **not refuted** overall, because an engine does clear 80/90 (Qwen, on diagram and rendered-table classes); but no engine clears 80/90 on **all** classes, and the mechanism of failure is the opposite of what was registered.

**Third clause (descriptions -> standard extraction prompt -> graph preservation >= 90%) is QUEUED BEHIND THE LLM TIER** - it requires the reserved campaign LLM and was not attempted with any reserved endpoint.

## Final report

In [11]:
import datetime
TS = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
def verdict_pair(m):
    return dict(recall=m['recall'], precision=m['precision'],
                recall_pass=m['recall']>=H218_RECALL_BAR, precision_pass=m['precision']>=H218_PREC_BAR)
report = {
 'round':'R21','hypotheses':['H218','H224'],'utc':TS,
 'corpus':'cpap-datasheets-and-manuals','gpu':'GPU0 RTX PRO 4000 Blackwell sm_120 (cu130 torch)',
 'frozen_inputs':{'census':CENSUS,'blind_gold':GOLD},
 'H224':{
   'bars':{'discard>=':H224_DISCARD_BAR,'false_discard<=':H224_FALSE_BAR,'refute_false>':H224_REFUTE_FALSE},
   **h224,
   'verdict':'CONFIRMED',
   'summary':'recurrence(>=24 placements)+sub-visible-size(<=98px) filter discards %.1f%% of the raw image stream at %.2f%% false-discard; recurrence+size carry the discrimination as predicted; entropy corroborates.'%(h224['discard_rate_place']*100, h224['false_discard_place']*100)
 },
 'H218':{
   'bars':{'recall>=':H218_RECALL_BAR,'precision>=':H218_PREC_BAR},
   'engines':{e:{c:verdict_pair(m) for c,m in scores[e].items()} for e in ENGINES},
   'hallucinations_qwen':{iid:d['hallucinated'] for iid,d in details['qwen2.5-vl-7b'].items() if d['hallucinated']},
   'fidelity_verdict':'PARTIAL PASS; not refuted (Qwen clears 80/90 on diagram+table); registered direction REFUTED (dense tables did not lose cells; product-photo precision failed via brand hallucination); Docling/SmolVLM fails recall on all classes',
   'extraction_clause':'QUEUED BEHIND LLM TIER (reserved campaign LLM; not attempted)'
 }
}
outp = f'reports/image-vlm-gates-r21-{TS}.json'
json.dump(report, open(outp,'w'), indent=1)
console.print(f'[bold green]wrote[/] {outp}')
console.print_json(json.dumps(report['H224']))
console.print_json(json.dumps(report['H218']['engines']))

/tmp/ipykernel_3094812/3767748490.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TS = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')


wrote reports/image-vlm-gates-r21-20260708T082746Z.json

{
  "bars": {
    "discard>=": 0.6,
    "false_discard<=": 0.05,
    "refute_false>": 0.1
  },
  "discard_rate_place": 0.9684628214492058,
  "discard_rate_record": 0.06167608286252354,
  "false_discard_place": 0.0,
  "false_discard_record": 0.0,
  "discarded_placements": 88840,
  "total_placements": 91733,
  "discarded_records": 131,
  "info_bearing_placements": 2262,
  "false_discarded_records": 0,
  "T_recur": 24,
  "T_tiny": 98,
  "entropy_ib_median": 3.73725769034542,
  "entropy_deco_median": 5.384605526134335,
  "verdict": "CONFIRMED",
  "summary": "recurrence(>=24 placements)+sub-visible-size(<=98px) filter discards 96.8% of the raw image stream at 0.00% false-discard; recurrence+size carry the discrimination as predicted; entropy corroborates."
}

{
  "qwen2.5-vl-7b": {
    "product_photo": {
      "recall": 0.906,
      "precision": 0.833,
      "recall_pass": true,
      "precision_pass": false
    },
    "diagram": {
      "recall": 0.977,
      "precision": 0.917,
      "recall_pass": true,
      "precision_pass": true
    },
    "rendered_table_vector": {
      "recall": 0.936,
      "precision": 1.0,
      "recall_pass": true,
      "precision_pass": true
    }
  },
  "docling-smolvlm-256m": {
    "product_photo": {
      "recall": 0.396,
      "precision": 1.0,
      "recall_pass": false,
      "precision_pass": true
    },
    "diagram": {
      "recall": 0.318,
      "precision": 1.0,
      "recall_pass": false,
      "precision_pass": true
    },
    "rendered_table_vector": {
      "recall": 0.617,
      "precision": 1.0,
      "recall_pass": false,
      "precision_pass": true
    }
  }
}

## Summary

- **H224 CONFIRMED** - deterministic recurrence + size pre-filter discards ~97% of the raw image stream at 0.00% false-discard, clearing the >= 60% discard and <= 5% false-discard bars with wide margin. Recurrence is the dominant feature; size handles the residual; entropy corroborates.
- **H218 PARTIAL PASS** - a local 7B VLM (Qwen2.5-VL) clears the 80/90 fidelity bars on diagrams and rendered tables and clears recall on product photos, but fails product-photo precision through brand hallucination; the lightweight Docling/SmolVLM captioner fails recall on every class. The registered "dense tables lose cells" prediction is refuted in direction. The extraction-preservation clause is queued behind the reserved LLM tier.